# Support Vector Machines

In the previous exercise you implemented and trained a **Perceptron**. Recall that the Perceptron learning algorithm finds a separating hyperplane by iteratively correcting misclassified points:
$$\mathbf{w} \leftarrow \mathbf{w} + y^* \mathbf{x}^*, \qquad b \leftarrow b + y^*,$$
where $\mathbf{x}^*$ is a misclassified point with true label $y^* \in \{-1, +1\}$. When data is linearly separable the algorithm converges — but it stops at the *first* valid boundary it finds. There are infinitely many valid hyperplanes and the Perceptron picks one arbitrarily.

**Support Vector Machines (SVMs)** resolve this ambiguity: among all separating hyperplanes, find the one that **maximises the margin** — the width of the empty "street" between the two classes. The boundary is uniquely determined, and only the training points closest to it (the **support vectors**) matter.

Normalising the hyperplane so that $y^{(i)}(\mathbf{w}^\intercal \mathbf{x}^{(i)} + b) \geq 1$ for all $i$, the geometric margin width is
$$\text{margin} = \frac{2}{\|\mathbf{w}\|}.$$
Maximising the margin is equivalent to solving
$$\min_{\mathbf{w},\,b}\; \frac{1}{2}\mathbf{w}^\intercal\mathbf{w} \qquad \text{s.t.} \quad y^{(i)}\bigl(\mathbf{w}^\intercal\mathbf{x}^{(i)}+b\bigr) \geq 1 \;\; \forall\, i,$$
a convex constraint optimization problem.

### Soft Margin and $C$

The hard-margin formulation requires perfectly separable data. In practice we use a **soft-margin** SVM that introduces slack variables $\zeta^{(i)} \geq 0$:
$$\min_{\mathbf{w},\,b,\,\boldsymbol{\zeta}}\; \frac{1}{2}\mathbf{w}^\intercal\mathbf{w} + C\sum_{i}\zeta^{(i)}, \qquad \text{s.t.}\quad y^{(i)}\bigl(\mathbf{w}^\intercal\mathbf{x}^{(i)}+b\bigr) \geq 1 - \zeta^{(i)}.$$

* **Small $C$** → wide margin, more violations tolerated → risk of underfitting
* **Large $C$** → narrow margin, fewer violations → risk of overfitting

### The Kernel Trick

When data cannot be separated by any hyperplane, SVMs can implicitly map features to a higher-dimensional space via a **kernel function** $K(\mathbf{a}, \mathbf{b}) = \phi(\mathbf{a})^\intercal\phi(\mathbf{b})$ — without ever computing $\phi$ explicitly:

| Kernel | $K(\mathbf{a}, \mathbf{b})$ | Key hyperparameters |
|---|---|---|
| Linear | $\mathbf{a}^\intercal\mathbf{b}$ | $C$ |
| Polynomial | $(\gamma\,\mathbf{a}^\intercal\mathbf{b}+r)^d$ | $C$, `degree` $d$, `coef0` $r$ |
| Gaussian RBF | $\exp(-\gamma\|\mathbf{a}-\mathbf{b}\|^2)$ | $C$, $\gamma$ |


In [ ]:
# @title Imports and plot settings — run this first!
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import ipywidgets as widgets
from ipywidgets import interact

from sklearn.datasets import make_classification, make_moons, load_digits, make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.svm import LinearSVC, SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score


np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})
print('Setup complete.')

In [ ]:
# @title Plotting helpers

C0, C1 = '#4CAF50', '#2196F3'
CMAP_BG = ListedColormap(['#d5f0d5', '#d0e8fb'])

def plot_dataset(X, y, ax, is_val=False, s=40):
    """Scatter a binary dataset. Train/Validation have distinct markers."""
    if not is_val:
        pairs = [(-1, C0, 'o', 'Train $-1$'), (1, C1, '^', 'Train $+1$')]
    else:
        pairs = [(-1, '#1b5e20', 'X', 'Validation $-1$'), (1, '#0d47a1', 'P', 'Validation $+1$')]
    for cls, col, mk, lbl in pairs:
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], c=col, marker=mk,
                   edgecolors='k', linewidths=0.5, s=s, zorder=3, label=lbl)

def plot_decision_boundary(clf, X_train, y_train, X_val, y_val, ax, title=''):
    """Plot filled decision regions. Features are used directly (no scaling)."""
    X_all = np.vstack([X_train, X_val]) if len(X_val) > 0 else X_train
    mg = 0.5
    xx, yy = np.meshgrid(
        np.linspace(X_all[:, 0].min() - mg, X_all[:, 0].max() + mg, 300),
        np.linspace(X_all[:, 1].min() - mg, X_all[:, 1].max() + mg, 300)
    )
    Z = (clf.predict(np.c_[xx.ravel(), yy.ravel()]) > 0).reshape(xx.shape).astype(float)
    ax.contourf(xx, yy, Z, cmap=CMAP_BG, alpha=0.35)
    ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=1.8)
    plot_dataset(X_train, y_train, ax, is_val=False)
    if len(X_val) > 0:
        plot_dataset(X_val, y_val, ax, is_val=True, s=60)
    ax.set_title(title, fontsize=10)

def draw_margin_lines(clf, X, ax):
    """Draw the ±1 margin hyperplanes for a LinearSVC (feature space, no scaling)."""
    if not hasattr(clf, 'coef_'):
        return
    w, b = clf.coef_[0], clf.intercept_[0]
    if abs(w[1]) < 1e-9:
        return
    x0 = np.linspace(X[:, 0].min() - 0.4, X[:, 0].max() + 0.4, 300)
    for offset, ls, lw, col in [(0, '-', 2., 'black'),
                                  (1, '--', 1.2, '#555'),
                                  (-1, '--', 1.2, '#555')]:
        x1 = (offset - b - w[0] * x0) / w[1]
        ax.plot(x0, x1, ls=ls, lw=lw, color=col)

def highlight_support_vectors(svc, X_train, ax):
    """Circle the support vectors. svc must be a fitted SVC (not LinearSVC)."""
    sv_idx = svc.support_
    ax.scatter(X_train[sv_idx, 0], X_train[sv_idx, 1],
               s=180, facecolors='none', edgecolors='red',
               linewidths=1.5, zorder=5, label='Support vectors')

def _lims(X_train, X_val):
    X_all = np.vstack([X_train, X_val])
    return ((X_all[:, 0].min() - 0.5, X_all[:, 0].max() + 0.5),
            (X_all[:, 1].min() - 0.5, X_all[:, 1].max() + 0.5))

# ── Per-task plot functions ────────────────────────────────────────────────────

def plot_raw_dataset(X_train, y_train, X_val, y_val, title='Binary dataset'):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    plot_dataset(X_train, y_train, ax, is_val=False)
    plot_dataset(X_val, y_val, ax, is_val=True)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_title(title)
    ax.legend(fontsize=8, loc='upper left')
    plt.tight_layout(); plt.show()

def plot_task1(perc, X_train, y_train, X_val, y_val, acc_tr, acc_val):
    xlim, ylim = _lims(X_train, X_val)
    fig, ax = plt.subplots(figsize=(6, 4.5))
    plot_decision_boundary(perc, X_train, y_train, X_val, y_val, ax,
        title=f'Task 1: Perceptron\nTrain Acc: {acc_tr:.0%} | Validation Acc: {acc_val:.0%}')
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.legend(fontsize=8, loc='upper left')
    plt.tight_layout(); plt.show()

def plot_task2(svm, X_train, y_train, X_val, y_val, acc_tr, acc_val):
    xlim, ylim = _lims(X_train, X_val)
    fig, ax = plt.subplots(figsize=(6, 4.5))
    plot_decision_boundary(svm, X_train, y_train, X_val, y_val, ax,
        title=f'Task 2: Hard Margin Linear SVM ($C=1000$)\nTrain Acc: {acc_tr:.0%} | Validation Acc: {acc_val:.0%}')
    draw_margin_lines(svm, X_train, ax)
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.legend(fontsize=8, loc='upper left')
    plt.tight_layout(); plt.show()

def plot_task3(perc, svm_lin, svc_lin, X_train, y_train, X_val, y_val):
    xlim, ylim = _lims(X_train, X_val)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True, sharex=True)
    plot_decision_boundary(perc, X_train, y_train, X_val, y_val, axes[0],
        title=f'Perceptron (Validation acc={perc.score(X_val, y_val):.0%})\narbitrary boundary')
    plot_decision_boundary(svm_lin, X_train, y_train, X_val, y_val, axes[1],
        title=f'Linear SVM, $C=1000$ (Validation acc={svm_lin.score(X_val, y_val):.0%})\nmaximum-margin boundary')
    draw_margin_lines(svm_lin, X_train, axes[1])
    highlight_support_vectors(svc_lin, X_train, axes[1])
    for ax in axes:
        ax.set_xlim(xlim); ax.set_ylim(ylim)
        ax.set_xlabel('$x_1$')
        ax.legend(fontsize=7, loc='upper left')
    axes[0].set_ylabel('$x_2$')
    plt.suptitle('Task 3: Perceptron vs Linear SVM', fontsize=12, y=1.02)
    plt.tight_layout(); plt.show()

def plot_task4(perc, svm_lin, X_train, y_train, X_val, y_val):
    xlim, ylim = _lims(X_train, X_val)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True, sharex=True)
    plot_decision_boundary(perc, X_train, y_train, X_val, y_val, axes[0],
        title=f'Perceptron (Validation acc: {perc.score(X_val, y_val):.0%})')
    plot_decision_boundary(svm_lin, X_train, y_train, X_val, y_val, axes[1],
        title=f'Linear SVM (Validation acc: {svm_lin.score(X_val, y_val):.0%})')
    for ax in axes:
        ax.set_xlim(xlim); ax.set_ylim(ylim)
        ax.set_xlabel('$x_1$')
        ax.legend(fontsize=7, loc='lower right')
    axes[0].set_ylabel('$x_2$')
    plt.suptitle('Task 4: Linear models on the Moons dataset', fontsize=12, y=1.02)
    plt.tight_layout(); plt.show()

print('Helpers loaded.')

def plot_task5(clf, X_train, y_train, X_val, y_val, degree, coef0, C, train_acc, val_acc):
    xlim, ylim = _lims(X_train, X_val)
    fig, ax = plt.subplots(figsize=(7, 5))
    plot_decision_boundary(clf, X_train, y_train, X_val, y_val, ax,
        title=f'Task 5: Poly SVM, degree={degree}, coef0={coef0}, $C={C:g}$\nTrain: {train_acc:.0%} | Validation: {val_acc:.0%}')
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.legend(fontsize=8, loc='lower right')
    plt.tight_layout(); plt.show()

def plot_task6(clf, X_train, y_train, X_val, y_val, gamma, C, train_acc, val_acc):
    xlim, ylim = _lims(X_train, X_val)
    fig, ax = plt.subplots(figsize=(7, 5))
    plot_decision_boundary(clf, X_train, y_train, X_val, y_val, ax,
        title=f'Task 6: RBF SVM, $\\gamma={gamma:g}$, $C={C:g}$\nTrain: {train_acc:.0%} | Validation: {val_acc:.0%}')
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.legend(fontsize=8, loc='lower right')
    plt.tight_layout(); plt.show()

---
## Part A: Linear SVM on Linearly Separable Data

We start with a clean, linearly separable binary dataset. Run the cell below to generate and visualise it.

### Generating the Data and Splitting

We draw samples from two 2D Gaussian distributions with well-separated means. Because a perfect linear separation exists, the decision boundary we are looking for is a **straight line** in 2D.

In [ ]:
X, y_01 = make_blobs(
    n_samples=100, centers=[[-2.5, -2.5], [3.5, 3.5]], cluster_std=1.6, random_state=42
)
y = np.where(y_01 == 0, -1, 1)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

# Note: for make_blobs the two features are on comparable scales, so scaling is not needed here.

print(f'Train n={len(y_train)}, Validation n={len(y_val)}')

In [ ]:
# @title Visualise the dataset
plot_raw_dataset(X_train, y_train, X_val, y_val, title='Linearly separable binary dataset')

### Task 1: Train a Perceptron

Fit scikit-learn's [`Perceptron`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Perceptron.html) on the training data.

Predict the accuracy on both the train and validation sets to calculate `acc_tr` and `acc_val`.

In [ ]:
# TODO: fit the perc_lin model
perc_lin = Perceptron(random_state=44)

#TODO: Compute accuracies
acc_tr  = None
acc_val = None

plot_task1(perc_lin, X_train, y_train, X_val, y_val, acc_tr, acc_val)

### Task 2: Train a Linear SVM

Fit a linear SVM model using [`LinearSVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html)  on the training data.

Also, compute the accuracy on both training and validation set for this model

In [ ]:
# TODO: fit the linear svm model
svm_lin = None

#TODO: Compute accuracies
acc_tr  = None
acc_val = None

plot_task2(svm_lin, X_train, y_train, X_val, y_val, acc_tr, acc_val)

### Task 3: Side-by-Side Comparison and Support Vectors

We will produce figures comparing the Perceptron and the Linear SVM.

To visualise the support vectors, fit a second SVM model using [`SVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) with a linear kernel. We need to use it instead of LinearSVC to highlight the support vectors.

**Questions to think about:**
* Both models may reach 100% training accuracy. Why might the SVM generalise better to unseen data?
* What would happen to the SVM boundary if you removed a training point that is *not* a support vector?

In [ ]:
# TODO: fit svc_lin on the training set
svc_lin = None

plot_task3(perc_lin, svm_lin, svc_lin, X_train, y_train, X_val, y_val)

print(f'Support vectors per class: {svc_lin.n_support_}')

---
## Part B: Non-Linearly Separable Data - The Moons Dataset

The true decision boundary here is **curved** - no straight line can correctly separate the two classes.

In [ ]:
X, y_01 = make_moons(n_samples=250, noise=0.20, random_state=42)
y = np.where(y_01 == 0, -1, 1)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

# Note: make_moons features are already on comparable scales, so scaling is not needed here.

print(f'Train n={len(y_train)}, Validation n={len(y_val)}')

In [ ]:
# @title Visualise the dataset
plot_raw_dataset(X_train, y_train, X_val, y_val, title='Moons dataset')

### Task 4: Perceptron and Linear SVM on Moons

Fit a `Perceptron` and a `LinearSVC` on the training dataset.

**Observation:** The Perceptron convergence theorem applies only to *linearly separable* data - what do you observe here? The soft-margin SVM always converges, but is the boundary the right *shape* for this data?

In [ ]:
# TODO: fit the perceptron and a linear svm
perc_moons = None

svm_lin_moons = None


plot_task4(perc_moons, svm_lin_moons, X_train, y_train, X_val, y_val)

### Task 5: Polynomial Kernel SVM

Since linear models are only so good, we need more expressive SVM models and we make use of kernels now. Complete the interactive_poly_svm function to fit polynomial SVM models for arbitrary degree, C and coef0. Make use of `SVC` method with a polynomial kernel.

The polynomial kernel $K(\mathbf{a},\mathbf{b}) = (\gamma\,\mathbf{a}^\intercal\mathbf{b}+r)^d$ implicitly computes all feature interactions up to degree $d$, allowing for curved boundaries.

Adjust the slider in the output to toggle degree, C and coef0. What do you observe? Make sure to wait as the plots take a few seconds to update.

In [ ]:
def interactive_poly_svm(degree, C, coef0):
    # TODO: Instantiate and fit a SVM with polynomial kernel
    svm_poly = None


    # TODO: Calculate accuracies
    train_acc = None
    val_acc   = None

    plot_task5(svm_poly, X_train, y_train, X_val, y_val, degree, coef0, C, train_acc, val_acc)

deg_slider   = widgets.IntSlider(value=3, min=1, max=10, step=1, description='Degree:')
c_slider     = widgets.FloatLogSlider(value=1.0, base=10, min=-3, max=3, step=0.5, description='C:')
coef0_slider = widgets.FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.5, description='coef0:')

_ = interact(interactive_poly_svm, degree=deg_slider, C=c_slider, coef0=coef0_slider)

### Task 6: Gaussian RBF Kernel SVM

Similarly, Complete the interactive_rbf function to fit polynomial SVM models for arbitrary gamma and C. Make use of `SVC` method with a rbf kernel. Adjust the slider in the output to toggle  C and gamma. What do you observe?

In [ ]:
def interactive_rbf(gamma, C):
    # TODO: Instantiate and fit a SVM with rbf kernel
    svm_rbf = None


    # TODO: Calculate accuracies
    train_acc = None
    val_acc   = None

    plot_task6(svm_rbf, X_train, y_train, X_val, y_val, gamma, C, train_acc, val_acc)

gamma_slider = widgets.FloatLogSlider(value=1.0, base=10, min=-2, max=2, step=0.25, description='Gamma:')
c_slider     = widgets.FloatLogSlider(value=1.0, base=10, min=-3, max=3, step=0.5, description='C:')

_ = interact(interactive_rbf, gamma=gamma_slider, C=c_slider)

---
## Going Further

* **Cross-validation for hyperparameter tuning:** use [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) or [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) to automatically find the best `C` and `gamma`.



* **Evaluation beyond accuracy:** explore [`classification_report()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html) for precision, recall and F1-score per class.
* **Fit a SVM model on MNIST dataset:** Make sure not to use the entire dataset for training as it can take a lot of time. RBF kernel works the best in practice.  


